# SAXS comparison: CRYSOL vs MAICoS vs experiment (BIFT-rescaled)



## Setup

In [ ]:
import os
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
import requests
from scipy.interpolate import griddata

from fairmd.idp.core import *
from fairmd.idp.databankLibrary import *
from fairmd.idp.databankio import *
from fairmd.idp.protein_functions import *

REPO_ROOT = "/Users/vrj897/Projects/idpdatabank"  # adjust if running elsewhere
databankPath = os.path.join(REPO_ROOT, "IDPdatabank") + "/"
os.environ["NMLDB_ROOT_PATH"] = databankPath

# BIFT setup 
# check the repo for more information https://github.com/ehb54/GenApp-BayesApp 
BIFT_EXEC = os.path.join(REPO_ROOT, "bift")
if not os.path.exists(BIFT_EXEC):
    print("BIFT executable not found. Trying to get it and install it...")
    response = requests.get("https://raw.githubusercontent.com/ehb54/GenApp-BayesApp/main/bin/source/bift.f")
    response.raise_for_status()
    with open("bift.f", "wb") as f:
        f.write(response.content)
    subprocess.run(["gfortran", "bift.f", "-march=native", "-O2", "-o", "bift"], check=True)
    subprocess.run(["mv", "bift", BIFT_EXEC], check=True)
    subprocess.run(["rm", "bift.f"], check=True)

PROCESSING_DIR = os.path.join(databankPath, "Scratch", "IDPensemble", "bift_processing")
os.makedirs(PROCESSING_DIR, exist_ok=True)

# Each list item is the parsed README.yaml of one simulation in the databank.
systems = initialize_databank()

## Readers

In [ ]:
def read_saxs_yaml(filename):
    """Read q[1/A], I(q), sd_I(q) from a SAXS.yaml / SAXS_MAICoS.yaml file."""
    with open(filename) as f:
        data = yaml.safe_load(f)
    I_KEYS = ("mean_I(q)[a.u.]", "mean_Inten[a.u.]", "I(q)[a.u.]")
    q, I, I_std = [], [], []
    for d in data:
        q_val = d.get("q[1/A]", d.get("q"))
        if q_val is None:
            continue
        I_val = next((d[k] for k in I_KEYS if k in d), None)
        if I_val is None:
            continue
        q.append(q_val)
        I.append(I_val)
        I_std.append(d.get("sd_I(q)[a.u.]", d.get("sd_Inten[a.u.]", 0.0)))
    return np.array(q), np.array(I), np.array(I_std)


def read_experimental_dat(filename):
    """Read q, I(q), sigma from a raw experimental SAXS .dat file (SASBDB format).
    Converts q to 1/A if it looks like it's in 1/nm (qmax > 2)."""
    q, I, sigma = [], [], []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line or not (line[0].isdigit() or line[0] in "+-."):
                continue
            parts = line.split()
            try:
                q_val, I_val = float(parts[0]), float(parts[1])
                sigma_val = float(parts[2]) if len(parts) >= 3 else np.nan
            except ValueError:
                continue
            q.append(q_val)
            I.append(I_val)
            sigma.append(sigma_val)
    q, I, sigma = np.array(q), np.array(I), np.array(sigma)
    sigma[sigma < 0] = np.nan
    good = ~np.isnan(sigma)
    if not good.any():
        raise ValueError(f"no usable sigma values in {filename}")
    if not good.all():
        idx = np.arange(len(sigma))
        sigma[~good] = np.interp(idx[~good], idx[good], sigma[good])
    if q.max() > 2.0:
        q = q / 10.0
    return q, I, sigma

## BIFT (experimental `.dat` files only)

In [ ]:
def run_bift(dat_path, label):
    """Run BIFT on one experimental .dat file, caching results under PROCESSING_DIR/<label>
    (label = experiment ID, e.g. 'SASDQK7'). Many simulated systems in the databank share
    the same experimental dataset, so this cache avoids re-running BIFT for the same experimental data.

    Returns the BIFT-rebinned (q, I, sigma) from rescale.dat, where sigma has already been
    rescaled by BIFT's error-correction factor -- the same 'scale_factor.dat' correction
    peptonebench applies in create_dataset_SAXS.ipynb, read here directly from rescale.dat
    since we don't need to reconcile it against a separately-curated .out/.json point grid
    the way that notebook does for raw SASBDB downloads -- we only have .dat files.
    """
    tmp_dir = os.path.join(PROCESSING_DIR, label)
    os.makedirs(tmp_dir, exist_ok=True)

    rescale_path = os.path.join(tmp_dir, "rescale.dat")
    scale_factor_path = os.path.join(tmp_dir, "scale_factor.dat")

    if not (os.path.exists(rescale_path) and os.path.exists(scale_factor_path)):
        q, I, sigma = read_experimental_dat(dat_path)
        np.savetxt(os.path.join(tmp_dir, "experimental.dat"), np.column_stack([q, I, sigma]))
        with open(os.path.join(tmp_dir, "inputfile.dat"), "w") as f:
            f.write("experimental.dat" + 17 * "\n")  # accept all BIFT defaults
        with open(os.path.join(tmp_dir, "inputfile.dat")) as stdin_f, \
             open(os.path.join(tmp_dir, "bift.log"), "w") as log_f:
            subprocess.run([BIFT_EXEC], cwd=tmp_dir, stdin=stdin_f, stdout=log_f,
                            stderr=subprocess.STDOUT, check=True)

    q_r, I_r, sigma_r = np.loadtxt(rescale_path, unpack=True)
    correction_factor = np.loadtxt(scale_factor_path)[0, 1]
    return q_r, I_r, sigma_r, correction_factor

## Metrics: RMSD , RMSE (`peptonebench`-style), reduced chi2

In [ ]:
def cubic_interp(x_new, x_known, y_known):
    """ this function helps to regrid the simulated curve onto the experimental curve's q-grid, using cubic interpolation."""
    return griddata(x_known, y_known, x_new, method="cubic")


def compute_rmsd(q_sim, I_sim, q_exp, I_exp):
    """Shape-comparison RMSD: fit experimental data onto the simulated curve via
    least-squares scale+offset over the overlapping q-range then RMSD / relative-RMSD of the residual."""
    q_min, q_max = max(q_sim.min(), q_exp.min()), min(q_sim.max(), q_exp.max())
    mask_sim = (q_sim >= q_min) & (q_sim <= q_max)
    if mask_sim.sum() < 3:
        return np.nan, np.nan, np.nan, np.nan
    I_exp_interp = cubic_interp(q_sim[mask_sim], q_exp, I_exp)
    A = np.vstack([I_exp_interp, np.ones_like(I_exp_interp)]).T
    scale, offset = np.linalg.lstsq(A, I_sim[mask_sim], rcond=None)[0]
    diff = I_sim[mask_sim] - (I_exp_interp * scale + offset)
    rmsd = np.sqrt(np.mean(diff**2))
    rel_rmsd = rmsd / np.mean(np.abs(I_sim[mask_sim]))
    return rmsd, rel_rmsd, scale, offset


def fit_sim_to_exp(q_sim, I_sim, q_exp, I_exp, sigma_exp):
    """Single-scale (no offset), sigma-weighted fit of a simulated curve onto the
    experimental curve's own (real, calibrated) intensity scale -- the Svergun et
    al. (1995) formula"""
    q_min, q_max = max(q_sim.min(), q_exp.min()), min(q_sim.max(), q_exp.max())
    mask_exp = (q_exp >= q_min) & (q_exp <= q_max)
    if mask_exp.sum() < 3:
        return 1.0
    I_sim_interp = cubic_interp(q_exp[mask_exp], q_sim, I_sim)
    sigma = sigma_exp[mask_exp]
    scale = np.sum((I_exp[mask_exp] / sigma) * (I_sim_interp / sigma)) / np.sum((I_sim_interp / sigma) ** 2)
    return scale


def compute_residual(q_sim, I_sim, scale, q_exp, I_exp, sigma_exp):
    """Error-weighted residual (I_exp - scale*I_sim) / sigma_exp, evaluated at the
    experimental q-points within the overlap of both curves' ranges (same overlap
    convention as fit_sim_to_exp/compute_rmse). Returns (q_used, residual) -- q_used
    is a subset of q_exp, since e.g. MAICoS doesn't cover the lowest experimental q
    values, so its residual is only defined where it actually has data."""
    q_min, q_max = max(q_sim.min(), q_exp.min()), min(q_sim.max(), q_exp.max())
    mask = (q_exp >= q_min) & (q_exp <= q_max)
    I_sim_on_exp = cubic_interp(q_exp[mask], q_sim, I_sim) * scale
    return q_exp[mask], (I_exp[mask] - I_sim_on_exp) / sigma_exp[mask]


def get_rmse(std_delta, expt_shift=None, weights=None):
    """Root Mean Square Error -- mirrors peptonebench.reweighting.get_RMSE exactly
    (Svergun et al. 1995 single-scale fit, for comparing arbitrary-unit simulated
    intensities to real-unit experimental intensities). Reimplemented locally instead
    of imported: importing peptonebench.reweighting pulls in mdtraj and other deps
    that aren't installed in the idpdatabank environment.

    std_delta: shape (n_samples, n_obs). Here n_samples=1 since we compare a single
    trajectory-averaged simulated curve, not a per-frame ensemble.
    """
    if std_delta.shape[1] == 0:
        return np.nan
    if weights is None:
        mean = np.average(std_delta, axis=0)
    else:
        mask_nan = np.isfinite(weights)
        if not mask_nan.any():
            return np.nan
        mean = np.average(std_delta[mask_nan], weights=weights[mask_nan], axis=0)
    if expt_shift is not None:
        c = np.sum(expt_shift * mean) / np.sum(mean**2)  # Svergun et al. (1995)
        mean = c * mean - expt_shift
    return np.linalg.norm(mean) / np.sqrt(len(mean))


def compute_rmse(q_sim, I_sim, q_exp, I_exp, sigma_exp):
    """Standardized RMSE, matching peptonebench's actual SAXS convention exactly
    (peptonebench/src/peptonebench/saxs.py::std_Igen_and_Iexp):
        std_delta = I_gen / sigma,  expt_shift = I_exp / sigma
    with no offset and no log transform, then scored by get_rmse() (Svergun et
    al. 1995 single-scale fit). peptonebench runs Pepsi-SAXS directly on the
    experimental q-grid so no interpolation is needed there; we instead
    interpolate the independently-computed CRYSOL/MAICoS curve onto that same
    experimental q-grid, so the residual is evaluated at the full experimental
    q-resolution rather than the coarser simulated-curve grid."""
    q_min, q_max = max(q_sim.min(), q_exp.min()), min(q_sim.max(), q_exp.max())
    mask_exp = (q_exp >= q_min) & (q_exp <= q_max)
    if mask_exp.sum() < 3:
        return np.nan
    I_sim_interp = cubic_interp(q_exp[mask_exp], q_sim, I_sim)
    std_delta = (I_sim_interp / sigma_exp[mask_exp])[None, :]
    expt_shift = I_exp[mask_exp] / sigma_exp[mask_exp]
    return get_rmse(std_delta, expt_shift)


def compute_chi2(q_sim, I_sim, q_exp, I_exp, sigma_exp):
    """Reduced chi-squared simulated curve onto the experimental curve's real scale with a full affine
    (scale AND offset) least-squares fit -- same 2-parameter form as that script's
    linear_model/curve_fit, here via lstsq -- evaluated on the full experimental
    q-grid (same convention as compute_rmse, via cubic_interp).

    chi2 = sum(((I_exp - fit) / sigma_exp)^2), reduced by (n_points - 1) degrees
    of freedom. Note: 5.regrid_and_scaling2curves.py divides by (n-1) even though
    it fits 2 parameters (a, b); this uses the statistically correct (n-2)."""
    q_min, q_max = max(q_sim.min(), q_exp.min()), min(q_sim.max(), q_exp.max())
    mask_exp = (q_exp >= q_min) & (q_exp <= q_max)
    n = mask_exp.sum()
    if n < 3:
        return np.nan
    I_sim_interp = cubic_interp(q_exp[mask_exp], q_sim, I_sim)
    A = np.vstack([I_sim_interp, np.ones_like(I_sim_interp)]).T
    scale, offset = np.linalg.lstsq(A, I_exp[mask_exp], rcond=None)[0]
    residual = (I_exp[mask_exp] - (I_sim_interp * scale + offset)) / sigma_exp[mask_exp]
    dof = n - 1
    return np.sum(residual**2) / dof if dof > 0 else np.nan

## Per-system evaluation + plotting

In [ ]:
def evaluate_system(system, plot=False):
    """Compare one databank system's SAXS.yaml (CRYSOL) and SAXS_MAICoS.yaml against
    every experimental SAXS dataset matched to it in its README.yaml. Returns a list
    of result rows (one per experimental dataset; usually 0-2 per system).

    Each metric is computed twice: '_raw' uses the experimental sigma exactly as
    reported in the .dat file (only -1 placeholders interpolated), '_bift' uses
    BIFT's rebinned, error-corrected data (rescale.dat) -- so the two can be
    compared directly to see what BIFT's rescaling actually changes."""
    sim_dir = os.path.join(databankPath, "Data", "Simulations", system["path"])
    saxs_file = os.path.join(sim_dir, "SAXS.yaml")
    maicos_file = os.path.join(sim_dir, "SAXS_MAICoS.yaml")
    if not (os.path.exists(saxs_file) and os.path.exists(maicos_file)):
        return []

    q_saxs, I_saxs, I_saxs_std = read_saxs_yaml(saxs_file)
    q_maicos, I_maicos, I_maicos_std = read_saxs_yaml(maicos_file)

    rows = []
    for exp_id in system["EXPERIMENT"]["saxs"]["path"]:
        exp_file = os.path.join(databankPath, "Data", "Experiments", "saxs", exp_id, "saxs.dat")
        if not os.path.exists(exp_file):
            continue
        try:
            q_raw, I_raw, sigma_raw = read_experimental_dat(exp_file)
        except ValueError as e:
            print(f"  skipping {exp_id}: {e}")
            continue
        try:
            q_bift, I_bift, sigma_bift, correction_factor = run_bift(exp_file, label=exp_id)
        except Exception as e:
            print(f"  BIFT failed for {exp_id}: {e}")
            continue

        row = {
            "SYSTEM": system["SYSTEM"],
            "TRJ": system["TRJ"][0][0] if system["TRJ"] else None,
            "ID": system["ID"],
            "EXPERIMENT": exp_id,
            "BIFT_correction_factor": correction_factor,
        }
        for sim_name, q_sim, I_sim in (("SAXS", q_saxs, I_saxs), ("MAICoS", q_maicos, I_maicos)):
            rmsd_raw, relrmsd_raw, *_ = compute_rmsd(q_sim, I_sim, q_raw, I_raw)
            rmsd_bift, relrmsd_bift, *_ = compute_rmsd(q_sim, I_sim, q_bift, I_bift)
            row[f"{sim_name}_RMSD_raw"] = rmsd_raw
            row[f"{sim_name}_relRMSD_raw"] = relrmsd_raw
            row[f"{sim_name}_RMSD_bift"] = rmsd_bift
            row[f"{sim_name}_relRMSD_bift"] = relrmsd_bift
            row[f"{sim_name}_RMSE_raw"] = compute_rmse(q_sim, I_sim, q_raw, I_raw, sigma_raw)
            row[f"{sim_name}_RMSE_bift"] = compute_rmse(q_sim, I_sim, q_bift, I_bift, sigma_bift)
            row[f"{sim_name}_chi2_raw"] = compute_chi2(q_sim, I_sim, q_raw, I_raw, sigma_raw)
            row[f"{sim_name}_chi2_bift"] = compute_chi2(q_sim, I_sim, q_bift, I_bift, sigma_bift)
        rows.append(row)

        if plot:
            plot_comparison(q_saxs, I_saxs, q_maicos, I_maicos, q_bift, I_bift, sigma_bift,
                             title=f"{system['SYSTEM']} {system['TRJ']} vs {exp_id}",
                             relrmsd_saxs=row["SAXS_relRMSD_bift"], relrmsd_maicos=row["MAICoS_relRMSD_bift"],
                             rmse_saxs=row["SAXS_RMSE_bift"], rmse_maicos=row["MAICoS_RMSE_bift"],
                             chi2_saxs=row["SAXS_chi2_bift"], chi2_maicos=row["MAICoS_chi2_bift"],
                             I_saxs_std=I_saxs_std, I_maicos_std=I_maicos_std,
                             q_raw=q_raw, I_raw=I_raw, sigma_raw=sigma_raw,
                             correction_factor=correction_factor)
    return rows


def plot_comparison(q_saxs, I_saxs, q_maicos, I_maicos, q_exp, I_exp, sigma_exp, title="",
                     relrmsd_saxs=None, relrmsd_maicos=None, rmse_saxs=None, rmse_maicos=None,
                     chi2_saxs=None, chi2_maicos=None, I_saxs_std=None, I_maicos_std=None,
                     correction_factor=None,
                     q_raw=None, I_raw=None, sigma_raw=None):
    """SAXS fit-quality plot. CRYSOL and MAICoS are each independently scale-fit
    (no offset) onto the experimental curve, then plotted on the same axes. The top panel is log-scale I(q) vs q,
    the bottom panel is linear-scale (I_exp - fit)/sigma vs q.

    Legend shows relRMSD, not RMSD: CRYSOL and MAICoS live on very different
    intensity scales, so raw RMSD isn't comparable between them -- relRMSD is.

    CRYSOL gets a shaded uncertainty band (its own frame-to-frame sd); MAICoS
    doesn't, since its YAML never stores one.

    Bottom panel: (I_exp - fit)/sigma at each experimental point, linear scale --
    MAICoS's residual only shows up where MAICoS actually has coverage.

    If given, raw (un-rescaled) error bars are drawn too, in another color, so
    BIFT's effect on the errors themselves is visible."""
    scale_saxs = fit_sim_to_exp(q_saxs, I_saxs, q_exp, I_exp, sigma_exp)
    scale_maicos = fit_sim_to_exp(q_maicos, I_maicos, q_exp, I_exp, sigma_exp)

    saxs_label = "CRYSOL (fit to experiment)"
    maicos_label = "MAICoS (fit to experiment)"
    if relrmsd_saxs is not None:
        saxs_label += f", relRMSD={relrmsd_saxs:.3g}"
    if rmse_saxs is not None:
        saxs_label += f", RMSE={rmse_saxs:.3g}"
    if chi2_saxs is not None:
        saxs_label += rf", $\chi^2_r$={chi2_saxs:.3g}"
    if relrmsd_maicos is not None:
        maicos_label += f", relRMSD={relrmsd_maicos:.3g}"
    if rmse_maicos is not None:
        maicos_label += f", RMSE={rmse_maicos:.3g}"
    if chi2_maicos is not None:
        maicos_label += rf", $\chi^2_r$={chi2_maicos:.3g}"

    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(7, 6.5), sharex=True,
                                    gridspec_kw={"height_ratios": [3, 1]})

    bift_label = "Experimental (BIFT-rescaled"
    bift_label += f", \u00d7{correction_factor:.3g})" if correction_factor is not None else ")"
    if q_raw is not None:
        ax0.errorbar(q_raw, I_raw, yerr=sigma_raw, fmt=".", markersize=4, alpha=0.15,
                     color="indianred", zorder=2.5, label="Experimental (raw sigma)")
    ax0.errorbar(q_exp, I_exp, yerr=sigma_exp, fmt=".", markersize=4, alpha=0.5,
                 color="grey", zorder=3, label=bift_label)
    (line_saxs,) = ax0.plot(q_saxs, I_saxs * scale_saxs, "-", linewidth=2.5, zorder=2, label=saxs_label)
    if I_saxs_std is not None and np.any(I_saxs_std > 0):
        ax0.fill_between(q_saxs, (I_saxs - I_saxs_std) * scale_saxs, (I_saxs + I_saxs_std) * scale_saxs,
                         color=line_saxs.get_color(), alpha=0.2, linewidth=0, zorder=1,
                         label="CRYSOL sd (frame-to-frame)")
    (line_maicos,) = ax0.plot(q_maicos, I_maicos * scale_maicos, "--", linewidth=2.5, zorder=1, label=maicos_label)
    if I_maicos_std is not None and np.any(I_maicos_std > 0):
        ax0.fill_between(q_maicos, (I_maicos - I_maicos_std) * scale_maicos, (I_maicos + I_maicos_std) * scale_maicos,
                         color=line_maicos.get_color(), alpha=0.2, linewidth=0, zorder=0,
                         label="MAICoS sd (frame-to-frame)")
    ax0.set_yscale("log")
    ax0.set_ylabel(r"$I(q)$")
    ax0.set_title(title)
    ax0.legend(frameon=False, fontsize=8)

    q_res_saxs, resid_saxs = compute_residual(q_saxs, I_saxs, scale_saxs, q_exp, I_exp, sigma_exp)
    q_res_maicos, resid_maicos = compute_residual(q_maicos, I_maicos, scale_maicos, q_exp, I_exp, sigma_exp)
    ax1.axhline(0, color="black", linewidth=1, zorder=0)
    ax1.plot(q_res_saxs, resid_saxs, ".", markersize=3, alpha=0.6, color=line_saxs.get_color(), zorder=2)
    ax1.plot(q_res_maicos, resid_maicos, ".", markersize=3, alpha=0.6, color=line_maicos.get_color(), zorder=1)
    ax1.set_xlabel("$q$ [1/\u00c5]")
    ax1.set_ylabel(r"$(I_{exp}-fit)/\sigma$")

    plt.tight_layout()
    plt.show()

## Run over the databank

In [ ]:
all_rows = []
for system in systems:
    all_rows.extend(evaluate_system(system, plot=True))

results = pd.DataFrame(all_rows)
results

## Summary

In [ ]:
print(f"{len(results)} (system, experiment) comparisons across {results['SYSTEM'].nunique()} unique proteins")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, suffix, tag in [(0, "raw", "raw experimental sigma"), (1, "bift", "BIFT-rescaled sigma")]:
    axes[row, 0].hist(results[f"SAXS_relRMSD_{suffix}"].dropna(), bins=20, alpha=0.7, label="SAXS (CRYSOL)")
    axes[row, 0].hist(results[f"MAICoS_relRMSD_{suffix}"].dropna(), bins=20, alpha=0.7, label="MAICoS")
    axes[row, 0].set_xlabel(f"relative RMSD ({tag})")
    axes[row, 0].legend(frameon=False)

    axes[row, 1].hist(results[f"SAXS_RMSE_{suffix}"].dropna(), bins=20, alpha=0.7, label="SAXS (CRYSOL)")
    axes[row, 1].hist(results[f"MAICoS_RMSE_{suffix}"].dropna(), bins=20, alpha=0.7, label="MAICoS")
    axes[row, 1].set_xlabel(f"RMSE (get_rmse, {tag})")
    axes[row, 1].legend(frameon=False)

    axes[row, 2].hist(results[f"SAXS_chi2_{suffix}"].dropna(), bins=20, alpha=0.7, label="SAXS (CRYSOL)")
    axes[row, 2].hist(results[f"MAICoS_chi2_{suffix}"].dropna(), bins=20, alpha=0.7, label="MAICoS")
    axes[row, 2].set_xlabel(f"reduced $\\chi^2$ ({tag})")
    axes[row, 2].legend(frameon=False)
plt.tight_layout()
plt.show()

results.sort_values("SAXS_RMSE_bift")

## RMSD / RMSE before vs. after BIFT rescaling

'raw' uses the experimental sigma as reported in the `.dat` file (only -1
placeholders interpolated); 'bift' uses BIFT's rebinned, error-corrected sigma
(`rescale.dat`). Points above the `y = x` line got a larger (worse) metric after
BIFT's correction -- i.e. BIFT determined the reported experimental errors were
overestimated, which tightens the effective error bars and inflates standardized
residuals such as RMSE.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
metric_pairs = [
    ("SAXS_RMSD_raw", "SAXS_RMSD_bift", axes[0, 0], "SAXS RMSD"),
    ("MAICoS_RMSD_raw", "MAICoS_RMSD_bift", axes[0, 1], "MAICoS RMSD"),
    ("SAXS_RMSE_raw", "SAXS_RMSE_bift", axes[1, 0], "SAXS RMSE"),
    ("MAICoS_RMSE_raw", "MAICoS_RMSE_bift", axes[1, 1], "MAICoS RMSE"),
    ("SAXS_chi2_raw", "SAXS_chi2_bift", axes[2, 0], "SAXS reduced chi2"),
    ("MAICoS_chi2_raw", "MAICoS_chi2_bift", axes[2, 1], "MAICoS reduced chi2"),
]
for col_raw, col_bift, ax, label in metric_pairs:
    paired = results[[col_raw, col_bift]].dropna()
    x, y = paired[col_raw], paired[col_bift]
    ax.scatter(x, y, alpha=0.7)
    if len(x):
        lims = [min(x.min(), y.min()), max(x.max(), y.max())]
        ax.plot(lims, lims, "k--", linewidth=1, label="y = x")
    ax.set_xlabel(f"{label} (raw sigma)")
    ax.set_ylabel(f"{label} (BIFT-rescaled sigma)")
    ax.set_title(label)
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()